<a href="https://colab.research.google.com/github/sispo3314/6th-BE-Blog/blob/main/PGPRDT_UniMiB_SHAR_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/datasets/UniMiB-SHAR"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ============================================================
# Physics-Guided Patch-Level Time/Frequency Dual Transformer
# ── UniMiB-SHAR ADL Dataset 적용 버전 ──
#
# 원본 모델(UCI-HAR)과의 차이점:
#   - 입력: (N, 128, 9) → (N, 151→152, 3)  (가속도계 3축만 사용)
#   - Physics descriptor: 9차원 → 6차원 (자이로/중력 관련 제거)
#   - 패딩: 151(소수) → 152 (patch_size=8로 나누기 위해)
#   - 클래스: 6 → 9 (ADL 9종)
# ============================================================

import os
import math
import copy
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, f1_score

# ============================================================
# 0) Reproducibility / Device
# ============================================================
SEED = 46
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ============================================================
# 1) Config
# ============================================================
BATCH_SIZE   = 128
EPOCHS       = 50
LR           = 1e-3
WEIGHT_DECAY = 5e-4
WARMUP_EP    = 5

# UniMiB: 151 timesteps → pad to 152 (8로 나누어 떨어지도록)
WINDOW_SIZE  = 152
PATCH_SIZE   = 8
N_PATCHES    = WINDOW_SIZE // PATCH_SIZE   # 19
NUM_FREQ_BANDS = 8

D_MODEL      = 64
NHEAD        = 4
NUM_LAYERS   = 2
D_FF         = 128
DROPOUT      = 0.1

GATE_HIDDEN        = 64
LABEL_SMOOTHING    = 0.1
LAMBDA_GATE        = 0.05        # alpha target loss weight

ACTIVITY_NAMES = [
    "StandingUpFL",      # 1 → 0
    "StandingUpFS",      # 2 → 1
    "Walking",           # 3 → 2
    "Running",           # 4 → 3
    "GoingUpS",          # 5 → 4
    "Jumping",           # 6 → 5
    "GoingDownS",        # 7 → 6
    "LyingDownFS",       # 8 → 7
    "SittingDown",       # 9 → 8
]

# ============================================================
# 2) UniMiB-SHAR Data Loader
# ============================================================
def load_unimib(data_path, train_ratio=0.8, seed=42):
    x_path = os.path.join(data_path, 'adl_data.npy')
    y_path = os.path.join(data_path, 'adl_labels.npy')

    if not os.path.exists(x_path):
        raise FileNotFoundError(f"File not found: {x_path}")

    raw_x = np.load(x_path)   # (7579, 453)
    raw_y = np.load(y_path)   # (7579, 3)

    # (N, 453) → (N, 3, 151) → (N, 151, 3)
    if raw_x.ndim == 2:
        raw_x = raw_x.reshape(-1, 3, 151).transpose(0, 2, 1)

    # 라벨: 첫 번째 컬럼(Action ID), 1~9 → 0~8
    if raw_y.ndim > 1:
        raw_y = raw_y[:, 0]
    y_all = (raw_y - 1).astype(np.int64)
    X_all = raw_x.astype(np.float32)

    # 패딩: 151 → 152 (마지막 timestep 복제)
    pad = X_all[:, -1:, :]
    X_all = np.concatenate([X_all, pad], axis=1)  # (N, 152, 3)

    # Train/Test 분할
    total_len = len(X_all)
    indices = np.arange(total_len)
    rng = np.random.RandomState(seed)
    rng.shuffle(indices)

    split_idx = int(total_len * train_ratio)
    train_idx = indices[:split_idx]
    test_idx  = indices[split_idx:]

    X_tr, y_tr = X_all[train_idx], y_all[train_idx]
    X_te, y_te = X_all[test_idx], y_all[test_idx]

    return X_tr, y_tr, X_te, y_te

print("\nLoading UniMiB-SHAR ADL...")
X_tr, y_tr, X_te, y_te = load_unimib(DATA_PATH)

print("Train:", X_tr.shape, y_tr.shape)
print("Test :", X_te.shape, y_te.shape)

NUM_CLASSES  = len(np.unique(y_tr))
NUM_CHANNELS = X_tr.shape[-1]   # 3

# [FIX 2] Physics descriptor는 raw signal에서 먼저 계산

# ============================================================
# 3) Physics Descriptor (가속도계 3축 전용)
#    UCI-HAR 9차원 → UniMiB 6차원
#    [acc_energy, jerk_energy, spectral_entropy,
#     dominant_freq_ratio, low_high_ratio, axis_correlation]
# ============================================================
def spectral_entropy_1d(x, eps=1e-8):
    fft_mag = np.abs(np.fft.rfft(x))
    psd = fft_mag ** 2
    psd = psd / (psd.sum() + eps)
    ent = -(psd * np.log(psd + eps)).sum()
    return float(ent / np.log(len(psd) + eps))

def dominant_freq_ratio_1d(x, eps=1e-8):
    fft_mag = np.abs(np.fft.rfft(x))
    power = fft_mag ** 2
    return float(power.max() / (power.sum() + eps))

def low_high_ratio_1d(x, eps=1e-8):
    power = np.abs(np.fft.rfft(x)) ** 2
    if len(power) < 4:
        return 1.0
    mid = max(1, len(power) // 3)
    low  = power[:mid].sum()
    high = power[mid:].sum()
    return float(low / (high + eps))

def safe_corr(a, b):
    a, b = np.asarray(a), np.asarray(b)
    if a.std() < 1e-8 or b.std() < 1e-8:
        return 0.0
    return float(np.corrcoef(a, b)[0, 1])

def extract_patch_physics_np(sample, patch_size=8):
    """
    sample: (152, 3) — accelerometer x, y, z
    return: (N_PATCHES, 6)
    """
    T, C = sample.shape
    N = T // patch_size
    feats = []

    for i in range(N):
        patch = sample[i*patch_size:(i+1)*patch_size]  # (P, 3)

        acc_mag = np.linalg.norm(patch, axis=-1)
        jerk = np.diff(acc_mag, prepend=acc_mag[0])

        acc_energy  = float(np.mean(acc_mag ** 2))
        jerk_energy = float(np.mean(jerk ** 2))
        spec_ent    = spectral_entropy_1d(acc_mag)
        dom_ratio   = dominant_freq_ratio_1d(acc_mag)
        low_high    = low_high_ratio_1d(acc_mag)

        axis_corrs = [
            safe_corr(patch[:, 0], patch[:, 1]),
            safe_corr(patch[:, 1], patch[:, 2]),
            safe_corr(patch[:, 0], patch[:, 2]),
        ]
        axis_correlation = float(np.mean(axis_corrs))

        feats.append([
            acc_energy, jerk_energy, spec_ent,
            dom_ratio, low_high, axis_correlation,
        ])

    return np.asarray(feats, dtype=np.float32)

def build_patch_physics_table(X, patch_size=8):
    all_feats = np.stack([extract_patch_physics_np(x, patch_size) for x in X]).astype(np.float32)
    mu  = all_feats.reshape(-1, all_feats.shape[-1]).mean(axis=0, keepdims=True)
    std = all_feats.reshape(-1, all_feats.shape[-1]).std(axis=0, keepdims=True) + 1e-8
    all_feats = (all_feats - mu[None, :, :]) / std[None, :, :]
    return all_feats, mu, std

print("\nBuilding patch-level physics descriptors (from raw signal)...")
phys_tr, phys_mu, phys_std = build_patch_physics_table(X_tr, PATCH_SIZE)
phys_te = np.stack([extract_patch_physics_np(x, PATCH_SIZE) for x in X_te]).astype(np.float32)
phys_te = (phys_te - phys_mu[None, :, :]) / phys_std[None, :, :]

PHYS_DIM = phys_tr.shape[-1]
print("Patch physics train:", phys_tr.shape)
print("Patch physics test :", phys_te.shape)
print("PHYS_DIM:", PHYS_DIM)

# [FIX 2] THEN normalize X for model input
mu  = X_tr.mean(axis=(0, 1), keepdims=True)
std = X_tr.std(axis=(0, 1), keepdims=True) + 1e-8
X_tr = (X_tr - mu) / std
X_te = (X_te - mu) / std

print("NUM_CLASSES :", NUM_CLASSES)
print("NUM_CHANNELS:", NUM_CHANNELS)
print("N_PATCHES   :", N_PATCHES)

# ============================================================
# 4) Dataset
# ============================================================
class UniMiBDataset(Dataset):
    def __init__(self, X, y, phys):
        self.X    = torch.FloatTensor(X)
        self.y    = torch.LongTensor(y)
        self.phys = torch.FloatTensor(phys)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.phys[idx], self.y[idx]

tr_ds = UniMiBDataset(X_tr, y_tr, phys_tr)
te_ds = UniMiBDataset(X_te, y_te, phys_te)

tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
te_loader = DataLoader(te_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ============================================================
# 5) Time / Frequency Patch Embedding
# ============================================================
class TimePatchEmbed(nn.Module):
    def __init__(self, num_channels, patch_size, d_model):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Linear(patch_size * num_channels, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        B, T, C = x.shape
        N = T // self.patch_size
        x = x[:, :N*self.patch_size, :]
        x = x.reshape(B, N, self.patch_size * C)
        x = self.proj(x)
        return self.norm(x)

class SpectralFilterbankEmbed(nn.Module):
    def __init__(self, num_channels, patch_size, num_bands, d_model):
        super().__init__()
        self.patch_size = patch_size
        self.num_channels = num_channels
        self.num_bins = patch_size // 2 + 1
        self.num_bands = num_bands

        self.filterbank = nn.Parameter(
            torch.randn(num_channels, self.num_bins, num_bands) * 0.02
        )
        self.proj = nn.Linear(num_channels * num_bands, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        B, T, C = x.shape
        N = T // self.patch_size
        x = x[:, :N*self.patch_size, :]
        x = x.reshape(B, N, self.patch_size, C)

        xf = torch.fft.rfft(x, dim=2)
        xf = torch.abs(xf)
        xf = xf.permute(0, 1, 3, 2)

        bands = torch.einsum('bncf,cfk->bnck', xf, self.filterbank)
        bands = bands.reshape(B, N, C * self.num_bands)

        z = self.proj(bands)
        return self.norm(z)

# ============================================================
# 6) Transformer Blocks
# ============================================================
class TransformerBlock(nn.Module):
    def __init__(self, d_model, nhead, d_ff, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=nhead,
            dropout=dropout, batch_first=True
        )
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x1 = self.norm1(x)
        z, _ = self.attn(x1, x1, x1)
        x = x + z
        x = x + self.ffn(self.norm2(x))
        return x

class BranchEncoder(nn.Module):
    def __init__(self, d_model, nhead, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, nhead, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        for blk in self.blocks:
            x = blk(x)
        return self.norm(x)

# ============================================================
# 7) Patch-Level Physics Gate
# ============================================================
class PatchPhysicsGate(nn.Module):
    def __init__(self, in_dim, hidden=64, temp_init=2.0):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, 1)
        )
        self.last_alpha_mean = 0.0

    def forward(self, z_phys_patch):
        logits = self.mlp(z_phys_patch)
        alpha = torch.sigmoid(logits)
        self.last_alpha_mean = alpha.mean().item()
        return alpha

# ============================================================
# 8) Full Model
# ============================================================
class PhysicsGuidedPatchDualTransformer(nn.Module):
    def __init__(
        self, num_channels, patch_size, num_patches, d_model, nhead,
        num_layers, d_ff, dropout, num_classes, phys_dim,
        gate_hidden=64, temp_init=2.0, num_freq_bands=8,
    ):
        super().__init__()
        self.num_patches = num_patches
        self.d_model = d_model

        self.time_embed = TimePatchEmbed(num_channels, patch_size, d_model)
        self.freq_embed = SpectralFilterbankEmbed(
            num_channels=num_channels, patch_size=patch_size,
            num_bands=num_freq_bands, d_model=d_model
        )

        self.time_pos = nn.Parameter(torch.zeros(1, num_patches, d_model))
        self.freq_pos = nn.Parameter(torch.zeros(1, num_patches, d_model))
        nn.init.trunc_normal_(self.time_pos, std=0.02)
        nn.init.trunc_normal_(self.freq_pos, std=0.02)

        self.time_encoder = BranchEncoder(d_model, nhead, d_ff, num_layers, dropout)
        self.freq_encoder = BranchEncoder(d_model, nhead, d_ff, num_layers, dropout)

        self.gate = PatchPhysicsGate(in_dim=phys_dim, hidden=gate_hidden, temp_init=temp_init)

        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model), nn.LayerNorm(d_model), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model, num_classes)
        )

    def forward(self, x, z_phys_patch):
        ht = self.time_embed(x) + self.time_pos
        hf = self.freq_embed(x) + self.freq_pos

        ht = self.time_encoder(ht)
        hf = self.freq_encoder(hf)

        alpha = self.gate(z_phys_patch)
        h_patch = alpha * ht + (1.0 - alpha) * hf

        hp_pool = h_patch.mean(dim=1)
        logits = self.classifier(hp_pool)
        return logits, alpha.squeeze(-1), ht, hf, h_patch

    def get_alpha_mean(self):
        return self.gate.last_alpha_mean

# ============================================================
# 9) Perturbation + differentiable patch physics (3ch 전용)
# ============================================================
def perturb_batch(x, noise_std=0.03, scale_std=0.05, bias_std=0.03):
    B, T, C = x.shape
    s = 1.0 + torch.randn(B, 1, C, device=x.device) * scale_std
    b = torch.randn(B, 1, C, device=x.device) * bias_std
    n = torch.randn_like(x) * noise_std
    return s * x + b + n

def safe_corr_torch(a, b, eps=1e-8):
    a = a - a.mean(dim=1, keepdim=True)
    b = b - b.mean(dim=1, keepdim=True)
    num = (a * b).sum(dim=1)
    den = torch.sqrt((a*a).sum(dim=1) + eps) * torch.sqrt((b*b).sum(dim=1) + eps)
    return num / (den + eps)

def patch_physics_torch(x, patch_size=8):
    """
    x: (B, T, 3) — accelerometer only
    return: (B, N, 6)
    """
    B, T, C = x.shape
    N = T // patch_size
    x = x[:, :N*patch_size, :].reshape(B, N, patch_size, C)

    acc_mag = torch.norm(x, dim=-1)
    jerk = torch.diff(acc_mag, dim=2, prepend=acc_mag[:, :, :1])

    acc_energy  = (acc_mag ** 2).mean(dim=2)
    jerk_energy = (jerk ** 2).mean(dim=2)

    acc_fft = torch.abs(torch.fft.rfft(acc_mag, dim=2)) ** 2
    acc_psd = acc_fft / (acc_fft.sum(dim=2, keepdim=True) + 1e-8)
    spec_ent = -(acc_psd * torch.log(acc_psd + 1e-8)).sum(dim=2) / math.log(acc_psd.shape[2] + 1e-8)

    dom_ratio = acc_fft.max(dim=2).values / (acc_fft.sum(dim=2) + 1e-8)

    mid = max(1, acc_fft.shape[2] // 3)
    low_high = acc_fft[:, :, :mid].sum(dim=2) / (acc_fft[:, :, mid:].sum(dim=2) + 1e-8)

    ax0 = safe_corr_torch(x[:,:,:,0].reshape(B*N, patch_size), x[:,:,:,1].reshape(B*N, patch_size)).reshape(B, N)
    ax1 = safe_corr_torch(x[:,:,:,1].reshape(B*N, patch_size), x[:,:,:,2].reshape(B*N, patch_size)).reshape(B, N)
    ax2 = safe_corr_torch(x[:,:,:,0].reshape(B*N, patch_size), x[:,:,:,2].reshape(B*N, patch_size)).reshape(B, N)
    axis_corr = (ax0 + ax1 + ax2) / 3.0

    z = torch.stack([
        acc_energy, jerk_energy, spec_ent,
        dom_ratio, low_high, axis_corr
    ], dim=-1)
    return z

# ============================================================
# [FIX 3] Alpha Target from Physics Descriptors (UniMiB: 6-dim)
# ============================================================
def make_alpha_target_from_physics(z_phys):
    """
    z_phys: (B, N, 6), standardized physics descriptors
    UniMiB descriptor indices:
      0 = acc_energy, 1 = jerk_energy, 2 = spectral_entropy,
      3 = dominant_freq_ratio, 4 = low_high_ratio, 5 = axis_correlation

    periodic_score = acc_energy + dom_ratio - spec_entropy
    high -> freq branch preferred -> low alpha
    low  -> time branch preferred -> high alpha
    """
    acc_energy   = z_phys[:, :, 0]
    spec_entropy = z_phys[:, :, 2]
    dom_ratio    = z_phys[:, :, 3]

    periodic_score = acc_energy + dom_ratio - spec_entropy

    freq_conf = torch.sigmoid(periodic_score)
    alpha_target = 1.0 - freq_conf
    alpha_target = 0.15 + 0.70 * alpha_target
    return alpha_target.unsqueeze(-1).clamp(0.15, 0.85)

# ============================================================
# 10) Train / Eval
# ============================================================
def train_model(model, tr_loader, te_loader, epochs, lr, warmup_ep, label,
                use_gate_target_loss=True, lambda_gate=LAMBDA_GATE):
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)

    def lr_lambda(ep):
        if ep < warmup_ep:
            return (ep + 1) / warmup_ep
        p = (ep - warmup_ep) / max(1, epochs - warmup_ep)
        return 0.5 * (1.0 + np.cos(np.pi * p))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    best_acc = 0.0
    best_state = None
    t0 = time.time()

    for ep in range(1, epochs + 1):
        model.train()
        tr_loss_sum = 0.0
        tr_corr = 0
        tr_n = 0

        for X_b, phys_b, y_b in tr_loader:
            X_b    = X_b.to(device)
            phys_b = phys_b.to(device)
            y_b    = y_b.to(device)

            optimizer.zero_grad()
            logits, alpha, _, _, _ = model(X_b, phys_b)
            ce_loss = criterion(logits, y_b)

            # [FIX 1] No balance loss
            # [FIX 3] Alpha target loss
            if use_gate_target_loss:
                alpha_target = make_alpha_target_from_physics(phys_b).detach()
                gate_loss = F.mse_loss(alpha.unsqueeze(-1), alpha_target)
                loss = ce_loss + lambda_gate * gate_loss
            else:
                loss = ce_loss

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            tr_loss_sum += ce_loss.item() * len(y_b)
            tr_corr += (logits.argmax(dim=1) == y_b).sum().item()
            tr_n += len(y_b)

        scheduler.step()

        model.eval()
        te_corr = 0
        te_n = 0
        with torch.no_grad():
            for X_b, phys_b, y_b in te_loader:
                X_b    = X_b.to(device)
                phys_b = phys_b.to(device)
                y_b    = y_b.to(device)
                logits, _, _, _, _ = model(X_b, phys_b)
                te_corr += (logits.argmax(dim=1) == y_b).sum().item()
                te_n += len(y_b)

        te_acc = te_corr / te_n
        if te_acc > best_acc:
            best_acc = te_acc
            best_state = copy.deepcopy(model.state_dict())

        if ep == 1 or ep % 5 == 0:
            print(
                f"[{label}] ep{ep:3d} | "
                f"loss={tr_loss_sum/max(tr_n,1):.4f} "
                f"tr={tr_corr/max(tr_n,1):.4f} "
                f"te={te_acc:.4f} best={best_acc:.4f} "
                f"alpha_mean={model.get_alpha_mean():.3f}"
            )

    print(f"[{label}] Done. Best={best_acc:.4f} ({time.time()-t0:.0f}s)")
    return best_acc, best_state

@torch.no_grad()
def evaluate_model(model, loader, label):
    model.eval()
    all_preds  = []
    all_labels = []
    all_alpha  = []

    alpha_by_class = {i: [] for i in range(NUM_CLASSES)}

    for X_b, phys_b, y_b in loader:
        X_b    = X_b.to(device)
        phys_b = phys_b.to(device)

        logits, alpha, _, _, _ = model(X_b, phys_b)
        preds = logits.argmax(dim=1).cpu().numpy()

        all_preds.extend(preds.tolist())
        all_labels.extend(y_b.numpy().tolist())
        all_alpha.extend(alpha.mean(axis=1).cpu().numpy().tolist())

        for a, y in zip(alpha.mean(axis=1).cpu().numpy(), y_b.numpy()):
            alpha_by_class[int(y)].append(float(a))

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_alpha  = np.array(all_alpha)

    print(f"\n{'='*70}")
    print(label)
    print(f"{'='*70}")
    print(classification_report(
        all_labels, all_preds,
        target_names=ACTIVITY_NAMES,
        digits=4, zero_division=0
    ))

    print("Routing statistics:")
    print(f"  mean alpha(time-branch weight): {all_alpha.mean():.4f}")
    print(f"  alpha std                     : {all_alpha.std():.4f}")
    print(f"  mean freq weight              : {(1-all_alpha).mean():.4f}")
    print()
    print("  Class-wise routing:")
    for c in range(NUM_CLASSES):
        arr = np.array(alpha_by_class[c]) if len(alpha_by_class[c]) > 0 else np.array([0.0])
        print(f"    {ACTIVITY_NAMES[c]:20s} alpha_mean={arr.mean():.4f}  alpha_std={arr.std():.4f}  freq_mean={(1-arr).mean():.4f}")

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    acc = (all_labels == all_preds).mean()
    return acc, macro_f1, all_preds, all_labels, all_alpha

# ============================================================
# 11) Main — 모델 생성 & 학습
# ============================================================
model = PhysicsGuidedPatchDualTransformer(
    num_channels=NUM_CHANNELS,
    patch_size=PATCH_SIZE,
    num_patches=N_PATCHES,
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    dropout=DROPOUT,
    num_classes=NUM_CLASSES,
    phys_dim=PHYS_DIM,
    gate_hidden=GATE_HIDDEN,
    num_freq_bands=NUM_FREQ_BANDS,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable params: {n_params:,}")

best_acc, best_state = train_model(
    model, tr_loader, te_loader,
    epochs=EPOCHS, lr=LR, warmup_ep=WARMUP_EP,
    label="PhysicsGuidedPatchDual",
    use_gate_target_loss=True,
    lambda_gate=LAMBDA_GATE,
)

model.load_state_dict(best_state)
acc, macro_f1, preds, labels, alphas = evaluate_model(
    model, te_loader,
    "Physics-Guided Patch-Level Time/Frequency Dual Transformer (UniMiB-SHAR)"
)

print(f"\nFinal Test Acc     : {acc:.4f}")
print(f"Final Test Macro-F1: {macro_f1:.4f}")
print("Done.")


Device: cuda

Loading UniMiB-SHAR ADL...
Train: (6063, 152, 3) (6063,)
Test : (1516, 152, 3) (1516,)

Building patch-level physics descriptors (from raw signal)...
Patch physics train: (6063, 19, 6)
Patch physics test : (1516, 19, 6)
PHYS_DIM: 6
NUM_CLASSES : 9
NUM_CHANNELS: 3
N_PATCHES   : 19

Trainable params: 149,698
[PhysicsGuidedPatchDual] ep  1 | loss=1.9019 tr=0.3191 te=0.3964 best=0.3964 alpha_mean=0.444
[PhysicsGuidedPatchDual] ep  5 | loss=0.7833 tr=0.8793 te=0.8160 best=0.8621 alpha_mean=0.359
[PhysicsGuidedPatchDual] ep 10 | loss=0.6295 tr=0.9477 te=0.9499 best=0.9505 alpha_mean=0.398
[PhysicsGuidedPatchDual] ep 15 | loss=0.5696 tr=0.9711 te=0.9578 best=0.9631 alpha_mean=0.427
[PhysicsGuidedPatchDual] ep 20 | loss=0.5351 tr=0.9873 te=0.9683 best=0.9723 alpha_mean=0.442
[PhysicsGuidedPatchDual] ep 25 | loss=0.5118 tr=0.9965 te=0.9796 best=0.9796 alpha_mean=0.451
[PhysicsGuidedPatchDual] ep 30 | loss=0.5065 tr=0.9974 te=0.9809 best=0.9809 alpha_mean=0.468
[PhysicsGuidedPatchD